# Single-grain material and shape comparison

Compare coercivity, remanence, squareness, and the standard second-quadrant maximum energy product across material/shape combinations. Numerical settings are filtered explicitly and are never averaged.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / 'utils').is_dir():
    SINGLE_GRAIN_DIR = Path('python/experiments/single_grain').resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from utils.plotting import (
    plot_hysteresis_curves, plot_material_metrics, plot_numerical_convergence,
    plot_property_space, plot_second_quadrant, save_figure,
)
from utils.reporting import (
    diagnostic_counts, material_summary, missing_combinations,
    select_representative_records,
)
from utils.results import filter_results, format_summary_table, load_results, write_summary_csv

# Configuration: change only this cell for a normal comparison.
EXCLUDED_RESULT_DIRS = {'res_15_06', 'res_16_06'}
_env_result_root = os.environ.get('SINGLE_GRAIN_RESULTS_ROOT')
if _env_result_root:
    RESULT_ROOTS = [Path(_env_result_root)]
else:
    RESULT_ROOTS = [
        path for path in sorted(SINGLE_GRAIN_DIR.glob('res_*'))
        if path.is_dir() and path.name not in EXCLUDED_RESULT_DIRS
    ]
RESOLUTION = 14
ADAPTIVE_DH_MIN_T = 0.001
PERIODIC_MODES = (False, True)
EXPECTED_SHAPES = None  # e.g. ('cube', 'sphere', 'ellipsoid_x', 'ellipsoid_z'); None means infer from data.
REPRESENTATIVE_SIZE_NM = 40.0
HYSTERESIS_FIELD_STEP_T = None  # None means use the smallest available adaptive field step.
HYSTERESIS_RESOLUTIONS = None  # None means use every available grid resolution.
EXPECTED_SIZES_NM = (10, 20, 40, 60, 80, 100, 200)
EXPORT = True
EXPORT_DIR = SINGLE_GRAIN_DIR / 'figures' / 'compare_material_properties'
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'font.size': 15,
    'axes.titlesize': 18,
    'axes.labelsize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'legend.title_fontsize': 13,
    'figure.titlesize': 20,
})

def format_for_powerpoint(fig, *, width=None, height=None):
    if width is not None or height is not None:
        fig.set_size_inches(
            width or fig.get_figwidth(),
            height or fig.get_figheight(),
            forward=True,
        )
    if fig._suptitle is not None:
        fig._suptitle.set_fontsize(20)
    for legend in fig.legends:
        for text in legend.get_texts():
            text.set_fontsize(12)
        if legend.get_title() is not None:
            legend.get_title().set_fontsize(13)
    for ax in fig.axes:
        ax.title.set_fontsize(18)
        ax.xaxis.label.set_fontsize(17)
        ax.yaxis.label.set_fontsize(17)
        ax.tick_params(axis='both', labelsize=14)
        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(12)
            if legend.get_title() is not None:
                legend.get_title().set_fontsize(13)
    return fig

## Load and validate the dataset

In [ ]:
def is_excluded_result_path(path):
    return any(part in EXCLUDED_RESULT_DIRS for part in Path(path).parts)

records, load_errors = load_results(RESULT_ROOTS)
records = [record for record in records if not is_excluded_result_path(record.path)]
print(f'Loaded {len(records)} result files from:')
for root in RESULT_ROOTS:
    print(f'  {Path(root).resolve()}')
if EXCLUDED_RESULT_DIRS:
    print(f'Excluded result directories: {sorted(EXCLUDED_RESULT_DIRS)}')
if load_errors:
    print()
    print(f'Skipped {len(load_errors)} malformed result file(s):')
    for path, error in load_errors[:10]:
        print(f'  {path}: {error}')
print()
print('Metric diagnostics:', diagnostic_counts(records))

In [ ]:
summary = material_summary(records)
if not summary:
    print('No single-grain results were found.')
else:
    print('material | shape | mu0 Ms [T] | A0 [J/m] | K0 [MJ/m3] | results | sizes [nm]')
    print('-' * 120)
    for row in summary:
        print(f"{row['material']:<18} {row['shape_variant']:<14} {row['mu0_Ms_T']:10.4g} "
              f"{row['A0_J_per_m']:11.4g} {row['K0_MJ_per_m3']:12.4g} "
              f"{row['result_count']:8d}  {row['sizes_nm']}")

## Select matching numerical settings and shapes

In [ ]:
selected_shapes = tuple(EXPECTED_SHAPES) if EXPECTED_SHAPES is not None else None
comparison = [
    record for record in filter_results(
        records, n=RESOLUTION, adaptive_dh_min_t=ADAPTIVE_DH_MIN_T,
        shapes=selected_shapes,
    )
    if record.periodic in PERIODIC_MODES
]
print(f'Selected {len(comparison)} results at n={RESOLUTION}, dh_min={ADAPTIVE_DH_MIN_T:g} T.')
print(format_summary_table(comparison[:20]))
if len(comparison) > 20:
    print(f'... plus {len(comparison) - 20} rows')

materials = sorted({record.material for record in comparison})
shapes = sorted(selected_shapes or {record.shape_variant for record in comparison})
print()
print(f'Shapes included: {shapes}')
missing = missing_combinations(
    comparison, materials=materials, shapes=shapes, sizes_nm=EXPECTED_SIZES_NM,
    resolutions=[RESOLUTION], dh_min_values=[ADAPTIVE_DH_MIN_T],
    periodic_modes=PERIODIC_MODES,
)
print()
print(f'Missing expected material/shape combinations: {len(missing)}')
for item in missing[:12]:
    print(' ', item)
if len(missing) > 12:
    print('  ...')

## Shape hysteresis curves at the highest field-step resolution

In [ ]:
hysteresis_groups = sorted(
    {
        (record.material, record.shape_variant)
        for record in records
        if selected_shapes is None or record.shape_variant in selected_shapes
    }
)
available_dh = sorted(
    {
        record.adaptive_dh_min_t for record in records
        if np.isfinite(record.adaptive_dh_min_t)
        and (selected_shapes is None or record.shape_variant in selected_shapes)
    }
)
hysteresis_dh_min = HYSTERESIS_FIELD_STEP_T if HYSTERESIS_FIELD_STEP_T is not None else (available_dh[0] if available_dh else None)
hysteresis_resolutions = tuple(HYSTERESIS_RESOLUTIONS or sorted({record.n for record in records}))

if hysteresis_dh_min is None:
    print('No adaptive field-step results are available for hysteresis plots.')
else:
    print(f'Hysteresis curves use dh_min={hysteresis_dh_min:g} T and include all available sizes.')
    for material, shape in hysteresis_groups:
        for periodic_mode in PERIODIC_MODES:
            has_mode_records = any(
                record.material == material
                and record.shape_variant == shape
                and record.periodic == periodic_mode
                and np.isclose(record.adaptive_dh_min_t, hysteresis_dh_min)
                for record in records
            )
            if not has_mode_records:
                continue

            fig, axes = plt.subplots(
                1, len(hysteresis_resolutions),
                figsize=(5.2 * len(hysteresis_resolutions), 4.4),
                squeeze=False,
                sharey=True,
            )
            plotted = False
            for ax, n in zip(axes.ravel(), hysteresis_resolutions):
                shape_records = sorted(
                    [
                        record for record in records
                        if record.material == material
                        and record.shape_variant == shape
                        and record.n == n
                        and np.isclose(record.adaptive_dh_min_t, hysteresis_dh_min)
                        and record.periodic == periodic_mode
                    ],
                    key=lambda record: (record.size_nm, str(record.path)),
                )
                if shape_records:
                    plot_hysteresis_curves(
                        shape_records,
                        ax=ax,
                        title=f'n={n}',
                        label_func=lambda record: f'{record.size_nm:g} nm',
                    )
                    plotted = True
                else:
                    ax.set(title=f'n={n}', xlabel=r'$\mu_0 H$ [T]', ylabel=r'$\mu_0 M_z$ [T]')
                    ax.text(0.5, 0.5, 'No matching result', ha='center', va='center', transform=ax.transAxes)
                    ax.grid(True, linestyle=':', alpha=0.6)
            if periodic_mode:
                fig.suptitle(f'{material} / {shape}: periodic hysteresis at highest field-step resolution')
            else:
                fig.suptitle(f'{material} / {shape}: hysteresis at highest field-step resolution')
            format_for_powerpoint(
                fig, width=6.4 * len(hysteresis_resolutions), height=4.5,
            )
            fig.tight_layout()
            if EXPORT and plotted:
                suffix = '_periodic' if periodic_mode else ''
                save_figure(fig, EXPORT_DIR / f'hysteresis_{material}_{shape}{suffix}.png')

## Four-shape hysteresis comparison at the highest spatial and field resolution

In [ ]:
SHAPE_PANEL_ORDER = ('cube', 'sphere', 'ellipsoid_x', 'ellipsoid_z')

for material in sorted({record.material for record in records}):
    material_records = [
        record for record in records
        if record.material == material
        and record.shape_variant in SHAPE_PANEL_ORDER
        and not record.periodic
    ]
    finite_dh = [
        record.adaptive_dh_min_t for record in material_records
        if np.isfinite(record.adaptive_dh_min_t)
    ]
    if not material_records or not finite_dh:
        print(f'No adaptive hysteresis results available for {material}.')
        continue

    highest_n = max(record.n for record in material_records)
    smallest_dh = min(finite_dh)
    fig, axes = plt.subplots(
        2, 2, figsize=(14, 10), sharex=True, sharey=True, squeeze=False,
    )
    plotted = False

    for ax, shape in zip(axes.ravel(), SHAPE_PANEL_ORDER):
        shape_records = sorted(
            [
                record for record in material_records
                if record.shape_variant == shape
                and record.n == highest_n
                and np.isclose(record.adaptive_dh_min_t, smallest_dh)
            ],
            key=lambda record: (record.size_nm, str(record.path)),
        )
        if shape_records:
            plot_hysteresis_curves(
                shape_records,
                ax=ax,
                title=shape.replace('_', ' '),
                label_func=lambda record: f'{record.size_nm:g} nm',
            )
            plotted = True
        else:
            ax.set(
                title=shape.replace('_', ' '),
                xlabel=r'$\mu_0 H$ [T]',
                ylabel=r'$\mu_0 M_z$ [T]',
            )
            ax.text(
                0.5, 0.5, 'No matching result',
                ha='center', va='center', transform=ax.transAxes,
            )
            ax.grid(True, linestyle=':', alpha=0.6)

    fig.suptitle(
        f'{material}: hysteresis by shape '
        f'(n={highest_n}, dH={smallest_dh:g} T)',
        fontsize=16,
    )
    format_for_powerpoint(fig, width=16, height=9)
    fig.tight_layout(rect=(0, 0, 1, 0.94))

    if EXPORT and plotted:
        safe_material = ''.join(
            char if char.isalnum() or char in {'-', '_'} else '_'
            for char in material
        ).strip('_')
        save_figure(
            fig,
            EXPORT_DIR / f'hysteresis_shapes_2x2_{safe_material}.png',
        )

## Coercivity, remanence, squareness, and maximum energy product by material and shape

In [ ]:
def export_safe_label(label):
    return ''.join(char if char.isalnum() or char in {'-', '_'} else '_' for char in label).strip('_')

if comparison:
    for material in sorted({record.material for record in comparison}):
        for periodic_mode in PERIODIC_MODES:
            mode_label = 'periodic' if periodic_mode else 'non-periodic'
            material_records = [
                record for record in comparison
                if record.material == material and record.periodic == periodic_mode
            ]
            if not material_records:
                print(f'No {mode_label} results available for {material}.')
                continue
            metrics_title = (
                f'{material}: single-grain permanent-magnet metrics '
                f'(n={RESOLUTION}, dh_min={ADAPTIVE_DH_MIN_T:g} T, {mode_label})'
            )
            metrics_fig, metrics_axes = plot_material_metrics(
                material_records,
                title=metrics_title,
                label_func=lambda record, periodic, n, dh_min: record.shape_variant,
            )
            reference_record = material_records[0]
            stoner_wohlfarth_hc_t = (
                2.0 * reference_record.K0_J_per_m3
                / reference_record.Ms_A_per_m
            )
            coercivity_ax = metrics_axes.ravel()[0]
            coercivity_ax.axhline(
                stoner_wohlfarth_hc_t,
                color='0.5',
                linestyle='--',
                linewidth=1.5,
                label=f'SW: {stoner_wohlfarth_hc_t:.2f}T',
            )
            coercivity_ax.legend(loc='best')
            format_for_powerpoint(metrics_fig, width=15, height=8)
            metrics_fig.tight_layout(rect=(0, 0, 1, 0.94))
            if EXPORT:
                safe_material = export_safe_label(material)
                safe_mode = export_safe_label(mode_label)
                save_figure(metrics_fig, EXPORT_DIR / f'metrics_vs_size_{safe_material}_{safe_mode}.png')
else:
    print('No matching results available for metric plots.')

## Numerical convergence (raw values, no averaging)

In [ ]:
comparison_groups = {(record.material, record.shape_variant) for record in comparison}
convergence_records = [
    record for record in records
    if (record.material, record.shape_variant) in comparison_groups
]
if convergence_records:
    for material in sorted({record.material for record in convergence_records}):
        material_records = [
            record for record in convergence_records
            if record.material == material
        ]
        convergence_fig, _ = plot_numerical_convergence(material_records)
        convergence_fig.suptitle(
            f'{material}: numerical convergence (raw values, no averaging)',
            fontsize=16,
        )
        format_for_powerpoint(
            convergence_fig,
            width=convergence_fig.get_figwidth() * 1.2,
        )
        convergence_fig.tight_layout(rect=(0, 0, 0.80, 0.94))
        if EXPORT:
            safe_material = export_safe_label(material)
            save_figure(
                convergence_fig,
                EXPORT_DIR / f'coercivity_convergence_{safe_material}.png',
            )
else:
    print('No matching results available for convergence plots.')

## Material/shape property space

In [ ]:
property_records = select_representative_records(
    comparison, target_size_nm=REPRESENTATIVE_SIZE_NM
)
if property_records:
    property_fig, property_ax = plt.subplots(figsize=(9, 5))
    plot_property_space(property_records, ax=property_ax)
    format_for_powerpoint(property_fig, width=9, height=5)
    property_fig.tight_layout()
    if EXPORT: save_figure(property_fig, EXPORT_DIR / 'property_space.png')
else:
    print('No matching results available for the property-space plot.')

## Optional export

In [ ]:
if EXPORT:
    csv_path = write_summary_csv(comparison, EXPORT_DIR / 'material_shape_metrics.csv')
    print(f'Exported figures and table to {EXPORT_DIR.resolve()}')
    print(f'CSV: {csv_path}')
else:
    print('EXPORT=False: no files were written. Set EXPORT=True in the configuration cell to save outputs.')

In [ ]:
MU0 = 4.0 * np.pi * 1e-7

# Fe16N2
#A0 = 7e-12
#Ms_Am = 2.4 / MU0
#K0 = 1e6 

# Sm2Fe17N3
A0 = 7e-12
Ms_Am = 1.54 / MU0
K0 = 8.9e6 


Hc_SW = 2 * K0 / (Ms_Am)
l_char = np.sqrt(A0 / (MU0 * Ms_Am**2 * 0.5))
print(f'Coercivity of an ideal Stoner-Wohlfarth particle with K={K0:.2e} J/m^3 and Ms={Ms_Am:.2e} A/m: {Hc_SW:.2f} T')
print(f'Characteristic length scale (sqrt(A/K)) for {material}: {l_char*1e9:.2f} nm')

In [ ]:
Ms_Am

In [ ]:
Ms_Am